# VoxCPM2 North Female Fine-Tune on Colab

Notebook này dùng chính các script đã chuẩn bị trong project để:

- setup môi trường Colab
- tải subset `nữ miền Bắc`
- build manifest train/val cho VoxCPM
- fine-tune `VoxCPM2` bằng `LoRA`
- test lại adapter sau khi train

Lưu ý thực tế:

- `VoxCPM2 LoRA` theo docs chính thức cần khoảng `20 GB VRAM`.
- Colab free thường không đủ VRAM. Notebook có preflight check để chặn sớm.

## 1. Mount Drive và cấu hình nguồn project

Chọn một trong hai cách:

- `SOURCE_MODE = "upload"`: upload `test-tts-colab-bundle.zip`
- `SOURCE_MODE = "git"`: clone repo Git của bạn

Nếu muốn tạo bundle từ máy local trước khi upload lên Colab, chạy:

```bash
bash make_colab_bundle.sh
```

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/voxcpm-colab')
PROJECT_ROOT = DRIVE_ROOT / 'test-tts'

SOURCE_MODE = 'upload'  # 'upload' or 'git'
GIT_URL = ''
BUNDLE_FILENAME = 'test-tts-colab-bundle.zip'
ALLOW_LOW_VRAM = True  # T4 (14.5 GB) đủ để chạy với config đã tối ưu

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('SOURCE_MODE =', SOURCE_MODE)
print('ALLOW_LOW_VRAM =', ALLOW_LOW_VRAM)

In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

if SOURCE_MODE == 'git':
    if not GIT_URL:
        raise ValueError('Set GIT_URL before running this cell.')

    git_dir = PROJECT_ROOT / '.git'
    if PROJECT_ROOT.exists() and not git_dir.exists():
        # Thư mục tồn tại nhưng không phải git repo (upload cũ) → xoá và clone mới
        print(f'[INFO] {PROJECT_ROOT} exists but is not a git repo. Removing and cloning fresh...')
        shutil.rmtree(PROJECT_ROOT)

    if PROJECT_ROOT.exists():
        print(f'[INFO] Pulling latest changes into {PROJECT_ROOT}...')
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
    else:
        print(f'[INFO] Cloning {GIT_URL} into {PROJECT_ROOT}...')
        subprocess.run(['git', 'clone', GIT_URL, str(PROJECT_ROOT)], check=True)

elif SOURCE_MODE == 'upload':
    from google.colab import files
    if PROJECT_ROOT.exists():
        print(f'{PROJECT_ROOT} already exists; skipping upload.')
    else:
        uploaded = files.upload()
        if not uploaded:
            raise ValueError('No file uploaded.')
        zip_name = BUNDLE_FILENAME if BUNDLE_FILENAME in uploaded else next(iter(uploaded))
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_name) as zf:
            zf.extractall(PROJECT_ROOT)
        os.remove(zip_name)
        print(f'Extracted {zip_name} into {PROJECT_ROOT}')
else:
    raise ValueError(f'Unsupported SOURCE_MODE={SOURCE_MODE!r}')

print('Done. PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
import os
os.chdir(PROJECT_ROOT)
print(os.getcwd())
!ls -la
!find colab -maxdepth 1 -type f | sort

## 2. Cài dependency vào Colab runtime

In [ ]:
!bash colab/bootstrap_colab.sh

## 3. Preflight GPU/VRAM

Mặc định notebook chặn nếu VRAM thấp hơn mức khuyến nghị cho `LoRA`.

In [ ]:
import subprocess

cmd = ['python3', 'colab/preflight_check.py', '--mode', 'lora']
if ALLOW_LOW_VRAM:
    cmd.append('--allow-low-vram')

result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Preflight failed with exit code {result.returncode}')

## 4. Tải subset `nữ miền Bắc` và build manifests

Cell này giúp bạn kiểm tra dữ liệu trước khi train.

In [ ]:
!python3 prepare_north_female_dataset.py
!python3 build_voxcpm_manifests.py
!cat datasets/cosrigel_north_female/stats.json
!echo '---'
!cat finetune/manifests/stats.json

## 5. Train LoRA

Checkpoint sẽ nằm ở `finetune/checkpoints/north_female_lora/latest`.
Nếu runtime reset, chỉ cần reconnect, vào lại thư mục Drive rồi chạy lại cell này để resume.

In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env['ALLOW_LOW_VRAM'] = '1' if ALLOW_LOW_VRAM else '0'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

steps = [
    ['python3', 'colab/preflight_check.py', '--mode', 'lora'] + (['--allow-low-vram'] if ALLOW_LOW_VRAM else []),
    ['python3', 'prepare_north_female_dataset.py'],
    ['python3', 'build_voxcpm_manifests.py'],
    # --refresh-upstream: force re-download upstream script để áp dụng gradient checkpointing patch
    ['python3', 'run_voxcpm_finetune.py',
     '--config', 'finetune/configs/voxcpm2_north_female_lora.yaml',
     '--allow-low-vram',
     '--refresh-upstream'],
]

step_names = ['preflight', 'prepare_dataset', 'build_manifests', 'finetune']

for name, cmd in zip(step_names, steps):
    print(f'\n{"="*60}')
    print(f'STEP: {name}')
    print(f'CMD:  {" ".join(cmd)}')
    print('='*60)
    result = subprocess.run(cmd, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0:
        print(f'[ERROR] Step "{name}" failed with exit code {result.returncode}', file=sys.stderr)
        raise SystemExit(f'Step "{name}" failed. See output above.')

## 6. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir finetune/tensorboard/north_female_lora

## 7. Test adapter mới train

In [ ]:
!bash colab/smoke_test_latest_lora_colab.sh
!ls -lh output_north_female_lora_colab.wav